# testPlayerData — exploring NFL player data with nflreadpy

[`nflreadpy`](https://github.com/nflverse/nflreadpy) is the Python version of nflverse's
`nflreadr`. It downloads public NFL datasets (player stats, rosters, snap counts, Next Gen Stats,
injuries and more) as **polars** DataFrames, cached locally after the first load.

This notebook goes through the player-related endpoints one at a time, then checks how
they connect to the ESPN league data.

**IDs.** nflverse identifies players by `gsis_id` (the NFL's ID, e.g. `00-0036355`). Player
stats call it `player_id`. `load_players()` and `load_ff_playerids()` map it to other platforms'
IDs, including `espn_id`, the same ID as the domain `Player.player_id`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import nflreadpy as nfl
import polars as pl

pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_rows(15)
pl.Config.set_fmt_str_lengths(40)

SEASON = nfl.get_current_season()
LAST_FULL_SEASON = SEASON - 1
print(f"nflreadpy {nfl.__version__} · current season {SEASON}, week {nfl.get_current_week()}")


def describe(df: pl.DataFrame, name: str) -> None:
    """Shape and column list of a dataset."""
    print(f"{name}: {df.height:,} rows × {df.width} columns")
    print(", ".join(df.columns))

## 1. Players: `load_players()`

One row per player who has ever appeared in nflverse data: biographical details, position,
draft info, current team, and IDs on other platforms.

In [ ]:
players = nfl.load_players()
describe(players, "players")

In [ ]:
players.filter(
    pl.col("display_name").is_in(["Bijan Robinson", "Ja'Marr Chase", "Josh Allen"])
).select(
    "display_name",
    "position",
    "latest_team",
    "gsis_id",
    "espn_id",
    "birth_date",
    "height",
    "weight",
    "college_name",
    "draft_year",
    "draft_round",
    "draft_pick",
    "years_of_experience",
)

## 2. ID crosswalk: `load_ff_playerids()`

DynastyProcess's mapping of one player across fantasy platforms (ESPN, Sleeper, Yahoo, MFL,
FantasyPros, PFF…). It's useful when a dataset uses an ID other than `gsis_id`, such as
FantasyPros rankings.

In [ ]:
ff_ids = nfl.load_ff_playerids()
describe(ff_ids, "ff_playerids")
ff_ids.select(
    "name",
    "position",
    "team",
    "gsis_id",
    "espn_id",
    "sleeper_id",
    "yahoo_id",
    "fantasypros_id",
    "pfr_id",
).head()

## 3. Weekly player stats: `load_player_stats(season)`

One row per player per game: passing, rushing, receiving, kicking and defensive stats, plus EPA
and nflverse's own `fantasy_points` (standard) and `fantasy_points_ppr`. `player_id` is the
`gsis_id`.

This league uses **half-PPR**, which is standard points plus 0.5 per reception (verified against
ESPN in section 13). nflverse's fantasy points don't include kicking: kickers show 0, although the
raw kicking stats (`fg_made`, `fg_made_40_49`, `pat_made`…) are there.

In [ ]:
weekly = nfl.load_player_stats(SEASON).with_columns(
    half_ppr=pl.col("fantasy_points") + 0.5 * pl.col("receptions")
)
describe(weekly, f"player_stats {SEASON} (weekly)")
print("weeks:", sorted(weekly["week"].unique().to_list()))

In [ ]:
# Top half-PPR performances of the season so far
weekly.filter(pl.col("position").is_in(["QB", "RB", "WR", "TE"])).sort(
    "half_ppr", descending=True
).select(
    "week",
    "player_display_name",
    "position",
    "team",
    "opponent_team",
    "passing_yards",
    "passing_tds",
    "rushing_yards",
    "rushing_tds",
    "receptions",
    "receiving_yards",
    "receiving_tds",
    "half_ppr",
).head(10)

## 4. Season totals: `load_player_stats(season, summary_level="reg")`

The same stats summed over the regular season, with usage shares such as `target_share` and
`air_yards_share`.

In [ ]:
season_stats = nfl.load_player_stats(LAST_FULL_SEASON, summary_level="reg").with_columns(
    half_ppr=pl.col("fantasy_points") + 0.5 * pl.col("receptions"),
)
describe(season_stats, f"player_stats {LAST_FULL_SEASON} (regular season)")
season_stats.filter(pl.col("position").is_in(["RB", "WR", "TE"])).sort(
    "half_ppr", descending=True
).select(
    "player_display_name",
    "position",
    "recent_team",
    "games",
    "carries",
    "targets",
    "target_share",
    "air_yards_share",
    "receptions",
    "receiving_yards",
    "rushing_yards",
    "half_ppr",
).head(10)

## 5. Weekly rosters: `load_rosters_weekly(season)`

Each team's roster every week, with roster status (active, injured reserve, practice squad…)
and many platform IDs.

In [ ]:
rosters = nfl.load_rosters_weekly(SEASON)
describe(rosters, f"rosters_weekly {SEASON}")
rosters.group_by("status").len().sort("len", descending=True)

## 6. Snap counts: `load_snap_counts(season)`

Offensive, defensive and special-teams snaps per player per game, as counts and percentages.
These use Pro Football Reference IDs (`pfr_player_id`). Map them through `players.pfr_id`.

In [ ]:
snaps = nfl.load_snap_counts(SEASON)
describe(snaps, f"snap_counts {SEASON}")
snaps.filter(pl.col("position").is_in(["RB", "WR", "TE"])).group_by(
    "player", "position", "team"
).agg(games=pl.len(), avg_offense_pct=pl.col("offense_pct").mean().round(2)).sort(
    "avg_offense_pct", descending=True
).head(10)

## 7. Expected fantasy points: `load_ff_opportunity(season)`

nflverse's opportunity model: given each player's usage (targets, carries, air yards, field
position), how many fantasy points *should* they have scored? `total_fantasy_points_exp` is
expected and `total_fantasy_points_diff` is actual minus expected. This is the player-level
version of the luck metric in `strengthOfSchedule`.

In [ ]:
opportunity = nfl.load_ff_opportunity(LAST_FULL_SEASON)
describe(opportunity, f"ff_opportunity {LAST_FULL_SEASON}")
opportunity.group_by("player_id", "full_name", "position").agg(
    games=pl.len(),
    actual=pl.col("total_fantasy_points").sum().round(1),
    expected=pl.col("total_fantasy_points_exp").sum().round(1),
    diff=pl.col("total_fantasy_points_diff").sum().round(1),
).filter(pl.col("games") >= 10).sort("diff", descending=True).head(10)

## 8. Next Gen Stats: `load_nextgen_stats(season, stat_type=...)`

NFL player-tracking metrics: separation, cushion and YAC over expected for receivers; time to
throw and air yards for passers; rush yards over expected for rushers. `week == 0` rows are
season totals.

In [ ]:
ngs = {t: nfl.load_nextgen_stats(SEASON, stat_type=t) for t in ("passing", "rushing", "receiving")}
for t, df in ngs.items():
    describe(df, f"nextgen_stats {t}")
    print()
ngs["receiving"].filter((pl.col("week") == 0) & (pl.col("targets") >= 30)).sort(
    "avg_separation", descending=True
).select(
    "player_display_name",
    "player_position",
    "team_abbr",
    "targets",
    "avg_separation",
    "avg_cushion",
    "avg_yac_above_expectation",
).head(10)

## 9. Pro Football Reference advanced stats: `load_pfr_advstats(season, stat_type=...)`

Charting stats from PFR, such as broken tackles, drops and yards before and after contact.
`stat_type` is `pass`, `rush`, `rec` or `def`; `summary_level` is `week` or `season`.

In [ ]:
pfr_rush = nfl.load_pfr_advstats(LAST_FULL_SEASON, stat_type="rush", summary_level="season")
describe(pfr_rush, f"pfr_advstats rush {LAST_FULL_SEASON}")
pfr_rush.head()

## 10. Injuries: `load_injuries(season)`

Weekly injury reports: game status and practice participation, keyed by `gsis_id`.

In [ ]:
injuries = nfl.load_injuries(SEASON)
describe(injuries, f"injuries {SEASON}")
latest_week = injuries["week"].max()
injuries.filter(
    (pl.col("week") == latest_week) & pl.col("position").is_in(["QB", "RB", "WR", "TE"])
).select(
    "week",
    "team",
    "full_name",
    "position",
    "report_primary_injury",
    "report_status",
    "practice_status",
).sort("report_status").head(15)

## 11. Depth charts: `load_depth_charts(season)`

Daily snapshots of every team's depth chart. It's large (hundreds of thousands of rows), so
filter to the latest snapshot. Keyed by both `gsis_id` and `espn_id`.

In [ ]:
depth = nfl.load_depth_charts(SEASON)
describe(depth, f"depth_charts {SEASON}")
latest = depth.filter(pl.col("dt") == pl.col("dt").max())
print("latest snapshot:", latest["dt"].max())
latest.filter((pl.col("team") == "ATL") & pl.col("pos_abb").is_in(["QB", "RB", "WR", "TE"])).sort(
    "pos_abb", "pos_rank"
).select("team", "pos_abb", "pos_rank", "player_name", "espn_id")

## 12. Rankings: `load_ff_rankings(type=...)`

FantasyPros expert consensus rankings (ECR). `type="week"` is this week's start/sit rankings;
`"draft"` is preseason. Keyed by `fantasypros_id`; map it through the ID crosswalk (section 2).

In [ ]:
rankings = nfl.load_ff_rankings("week")
describe(rankings, "ff_rankings week")
rankings.filter(pl.col("pos") == "RB").sort("ecr").select(
    "pos_rank",
    "player_name",
    "team",
    "player_opponent",
    "ecr",
    "best",
    "worst",
    "sd",
    "start_sit_grade",
).head(10)

## 13. Connecting to the ESPN league

The domain `Player.player_id` is ESPN's player ID, and `players.espn_id` maps it to `gsis_id`.
Two checks:

1. **Coverage:** what share of the league's rostered players find an nflverse match?
2. **Scoring:** does half-PPR from nflverse stats reproduce ESPN's points? Checked for QB, RB,
   WR and TE. Kicker points would have to be computed from the kicking stats with the league's
   kicking rules, since nflverse's fantasy points leave kicking out.

In [ ]:
from fantasy_analysis.application.frames import lineup_entries_frame
from fantasy_analysis.config import settings
from fantasy_analysis.infrastructure.espn import EspnLeagueRepository

league = EspnLeagueRepository.from_settings(settings).get_league(SEASON)
entries = pl.from_pandas(lineup_entries_frame(league))

espn_to_gsis = players.select(
    pl.col("espn_id").cast(pl.Int64, strict=False).alias("player_id"), "gsis_id"
).drop_nulls()
entries = entries.join(espn_to_gsis, on="player_id", how="left")

entries.group_by("position").agg(
    lineup_entries=pl.len(), matched=pl.col("gsis_id").is_not_null().mean().round(3)
).sort("position")

Team defenses (DST) never match. nflverse has no player rows for team defenses, so D/ST
points would come from team-level data (`load_team_stats`) instead. Every other position should
match fully.

In [ ]:
scored = (
    entries.filter(pl.col("position").is_in(["QB", "RB", "WR", "TE"]))
    .join(
        weekly.select(pl.col("player_id").alias("gsis_id"), "week", "half_ppr"),
        on=["gsis_id", "week"],
        how="inner",
    )
    .with_columns(diff=(pl.col("points") - pl.col("half_ppr")).round(2))
)

print(f"{scored.height} player-weeks compared")
print(f"exact match (±0.05): {(scored['diff'].abs() < 0.05).mean():.1%}")
scored.filter(pl.col("diff").abs() >= 0.05).select(
    "week", "team", "player", "position", "points", "half_ppr", "diff"
).head(10)

## 14. Player stats on the domain `Player`

The project now has this built in: `NflversePlayerStatsRepository` loads the weekly stats, and
`attach_player_stats` puts them on each league player as `player.stats[week]`, a `PlayerStats`
object. Every lineup entry for the same player shares the same stats.

In [ ]:
from fantasy_analysis.application.frames import player_stats_frame
from fantasy_analysis.application.nfl_stats import attach_player_stats
from fantasy_analysis.infrastructure.nflverse import NflversePlayerStatsRepository

attach_player_stats(league, NflversePlayerStatsRepository())

# One player's season, straight from the domain object
team = league.teams[0]
entry = team.lineups[1].starters[0]
player = entry.player
print(f"{player.name} ({player.position}, {player.pro_team.abbreviation}) on {team.name}")
for week, stats in sorted(player.stats.items()):
    print(
        f"  week {week}: vs {stats.opponent.abbreviation}, {stats.receptions} rec, "
        f"{stats.receiving_yards + stats.rushing_yards} yds, {stats.half_ppr_points:.1f} pts"
    )

In [ ]:
# Every rostered player's weekly stats as a DataFrame
stats_df = player_stats_frame(league)
print(stats_df.shape)
stats_df.head()

# Kickers: half_ppr_points is 0 (nflverse doesn't score kicking); espn_points has the real score
# kicking = ["player", "week", "fg_made", "pat_made", "espn_points"]
# stats_df.loc[stats_df["position"] == "K", kicking]

## 15. NFL teams and team stats

Every NFL team is an `NFLTeam` object, one per team for the season (`league.nfl_teams`).
`player.pro_team`, `stats.team` and `stats.opponent` all point at these objects.
`attach_team_stats` fills `NFLTeam.stats[week]` with an `NFLTeamStats`: offense, defense and
special teams, plus points allowed and yards allowed. **A D/ST's stats are its team's:**
`player.pro_team.stats[week]`.

In [ ]:
from fantasy_analysis.application.frames import nfl_team_stats_frame
from fantasy_analysis.application.nfl_stats import attach_team_stats
from fantasy_analysis.infrastructure.nflverse import NflverseTeamStatsRepository

attach_team_stats(league, NflverseTeamStatsRepository())

# A D/ST from the league, straight from the domain objects
dst_entry = next(
    e for t in league.teams for e in t.lineups[1].starters if e.player.position == "DST"
)
dst_team = dst_entry.player.pro_team
print(f"{dst_entry.player.name}: {len(dst_team.stats)} games with stats")
for week, s in sorted(dst_team.stats.items()):
    print(
        f"  week {week}: vs {s.opponent.abbreviation}, allowed {s.points_allowed} pts / "
        f"{s.yards_allowed} yds, {s.sacks:g} sacks, {s.takeaways} takeaways, "
        f"{s.defensive_tds + s.special_teams_tds} TDs"
    )

In [ ]:
# Every NFL team's weekly stats; dst_espn_points is filled for weeks its D/ST was rostered
team_stats_df = nfl_team_stats_frame(league)
print(team_stats_df.shape)
team_stats_df.head()